# Strong vs. weak boundary and initial conditions in `Scimba`

In this second tutorial, we will see how to approximate the solutions of the problems of the previous tutorial using PINNs with **strong** boundary conditions.

Consider a PDE $\mathcal{D}[u] = 0$
on a domain $\Omega$ with boundary $\partial \Omega$,
with Dirichlet boundary conditions $u = g$ on $\partial \Omega$.

Classically, PINNs satisfy boundary conditions in a *weak* way:
the loss function is
$$\mathcal{L}(\theta) = \int_{\Omega} \mathcal{D}[u_\theta](x) dx + \lambda \int_{\partial \Omega} (u_\theta - g)^2 ds.$$
It may be quite complicated to tune the parameter $\lambda$ to ensure that the boundary conditions are satisfied with sufficient accuracy.
Instead, one may use a *strong* enforcement of the boundary conditions, by defining a new function $v_\theta$ such that $v_\theta = g$ on $\partial \Omega$, and then minimizing the loss function
$$\mathcal{L}(\theta) = \int_{\Omega} \mathcal{D}[v_\theta](x) dx.$$
In this case, the boundary conditions are satisfied exactly, and the loss function is simpler to minimize.
Of course, one has to find a suitable function $v_\theta$ that satisfies the boundary conditions, which may be non-trivial in some cases.
For simple rectangle domains like $[0, 1]$, one may use a function of the form $v_\theta(x) = g(x) + x (1 - x) u_\theta(x)$, which satisfies $v_\theta(0) = g(0)$ and $v_\theta(1) = g(1)$. In general, for Dirichlet boundary conditions, $v_\theta$ would be a level-set function of the domain $\Omega$.

In general, we call this procedure a **post-processing** of the PINN, and we see the function $v_\theta$ as
$$v_\theta = \mathcal{A} \circ u_\theta,$$
where $\mathcal{A}$ is an operator that enforces the boundary conditions.
We also support **pre-processing** of the PINN, e.g. for coordinate transforms.
Hence, the full pipeline is $v_\theta = \mathcal{A} \circ u_\theta \circ \mathcal{B}$,
where $\mathcal{A}$ and $\mathcal{B}$ are the post- and pre-processing operators, respectively, both defaulting to the identity.


## 1/ Approximation of the solution of a parametric 2D Laplacian

We consider the equation:

$$- \mu \Delta u = 2(\mu\pi)^2 \sin(\pi x) \sin(\pi y)$$

on $(x,y) \in \Omega := [-1, 1]\times[-1, 1]$ with parameter $\mu\in \mathbb{M}=[1,2]$ subject to the Dirichlet condition:

$$ u = 0 $$

on $\partial\Omega$.

The exact solution is given by

$$ u(x,y,\mu) = \mu \sin(\pi x) \sin(\pi y). $$

Hence, we are essentially solving a **three-dimensional problem**, with two space dimensions and one parameter dimension. The solution is a function of three variables, $u(x,y,\mu)$. We emphasize that we solve this problem *in a monolithic fashion*, without snapshots on $\mu$, on the whole continuous interval $\mu \in \mathbb{M}=[1,2]$.

Let us first import the necessary modules, and then define some global variables, the right-hand side and the analytic solution.


In [ ]:
# # uncomment the line below if you are on Google Colab
# !pip install scimba

import timeit
import jax
import jax.numpy as jnp

from scimba_jax.domains.meshless_domains.domains_2d import Square2D
from scimba_jax.nonlinear_approximation.integration.monte_carlo import (
    DomainSampler,
    TensorizedSampler,
)
from scimba_jax.nonlinear_approximation.integration.monte_carlo_parameters import (
    UniformParametricSampler,
)
from scimba_jax.physical_models.elliptic_pde.laplacians import (
    ParametricLaplacianDirichletND,
)
from scimba_jax.nonlinear_approximation.networks.mlp import MLP
from scimba_jax.nonlinear_approximation.approximation_spaces.approximation_spaces import (
    ApproximationSpace,
)
from scimba_jax.nonlinear_approximation.numerical_solvers.projectors import Projector

## Student version: a notebook with holes

This notebook is the **student version** of `solutions/2a_boundary_conditions.ipynb`: some pieces of
code have been removed and replaced by calls to `TODO("...")`.

Running a cell that still contains a `TODO(...)` raises a `NotImplementedError`
with a hint. Replace each `TODO(...)` with your own code then re-run the cell.
The surrounding text usually contains the answer!

The complete, filled notebook is `solutions/2a_boundary_conditions.ipynb`.


In [ ]:
def TODO(hint: str = ""):
    """Placeholder for a hole to fill in: raises a clear error when executed."""
    raise NotImplementedError("hole to fill" + (f": {hint}" if hint else ""))

In [ ]:
N_COLLOC = 1000  # the number of collocation points for the interior residual
N_EPOCHS = 50  # the number of steps of optimization
S_LAYERS = [16, 16]  # the hidden layers widths of the MLPs defined below


def f_rhs(xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Right-hand side of the PDE."""
    x, y = xy[..., 0:1], xy[..., 1:2]
    return 2 * (mu[..., 0:1] * jnp.pi) ** 2 * jnp.sin(jnp.pi * x) * jnp.sin(jnp.pi * y)


def analytic_solution(xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Exact solution of the PDE."""
    x, y = xy[..., 0:1], xy[..., 1:2]
    return mu * jnp.sin(jnp.pi * x) * jnp.sin(jnp.pi * y)

Let us now define a PINN using **strong** boundary conditions.

First define the domains and the sampler; notice the `bc=False` argument of the sampler.
It tells the sampler not to construct the boundary collocation points, as they will not be used in the loss function, since we will enforce the boundary conditions in a strong way.


In [ ]:
domain_x = Square2D([(-1.0, 1.0), (-1.0, 1.0)], is_main_domain=True)

domain_mu = [(1.0, 2.0)]
sampler = TensorizedSampler(
    [DomainSampler(domain_x), UniformParametricSampler(domain_mu)],
    bc=TODO("with strong BC, boundary collocation points are not needed: bc=False"),
)

Use argument `bc="strong"` when defining the physical model:


In [ ]:
model = ParametricLaplacianDirichletND(
    domain_x,
    f_rhs,
    bc=TODO("\"strong\" enforcement of the boundary conditions"),
)

To enforce the boundary conditions strongly, we apply a post-processing to the evaluation of the PINN:


In [ ]:
def post_processing(
    approx: jnp.ndarray, xy: jnp.ndarray, mu: jnp.ndarray
) -> jnp.ndarray:
    r"""Post-processing function to enforce the boundary conditions.

    We write v_theta = A \circ u_theta, where
    A(x, y) = (x + 1)(1 - x)(y + 1)(1 - y)
    is a function that enforces the boundary conditions,
    and u_theta is the output of the neural network.
    """
    x, y = xy[0:1], xy[1:2]
    TODO("multiply the network output by a function that vanishes on the boundary of [-1, 1]^2, e.g. (x + 1)(1 - x)(y + 1)(1 - y)")

Generate the approximation space using this post-processing:


In [ ]:
key = jax.random.PRNGKey(0)  # initialize the random generator
nn = MLP(in_size=3, out_size=1, hidden_sizes=S_LAYERS, key=key)
space = ApproximationSpace(
    {"x": 2, "mu": 1},  # x has dim 2, mu has dim 1
    [(nn, "scalar", None)],  # nn models a scalar function u
    model_type="x_mu",  # u depends on x and mu
    post_processing=post_processing,  # equip the space with the post-processing
)

Finally, define the PINN and train it as previously:


In [ ]:
pinn = Projector(model, space, sampler)

start = timeit.default_timer()
key, pinn = pinn.project(key, space, N_EPOCHS, N_COLLOC)
end = timeit.default_timer()

print(f"time for {N_EPOCHS} epochs: {end - start:.2f} s")

print("interior loss after training: ", pinn.best_loss["interior"])
print("total    loss after training: ", pinn.best_loss["total"])

As a reference, let us define and train a PINN with **weak** boundary conditions (as in the previous tutorial):


In [ ]:
N_BC_COLLOC = 1000  # the number of collocation points for the boundary residual

sampler = TensorizedSampler(
    [DomainSampler(domain_x), UniformParametricSampler(domain_mu)],
    bc=TODO("with weak BC, boundary collocation points are needed: bc=True"),
)

model = ParametricLaplacianDirichletND(
    domain_x,
    f_rhs,
    bc=TODO("here the classical weak enforcement: bc=\"weak\""),
)

key = jax.random.PRNGKey(0)  # initialize the random generator
nn2 = MLP(in_size=3, out_size=1, hidden_sizes=S_LAYERS, key=key)
space2 = ApproximationSpace(
    {"x": 2, "mu": 1},  # x has dim 2, mu has dim 1
    [(nn2, "scalar", None)],  # nn2 models a scalar function u
    model_type="x_mu",  # u depends on x and mu
)

pinn2 = Projector(model, space2, sampler, matrix_regularization=1e-5)

start = timeit.default_timer()
key, pinn2 = pinn2.project(key, space2, N_EPOCHS, N_COLLOC, N_BC_COLLOC)
end = timeit.default_timer()

print(f"time for {N_EPOCHS} epochs: {end - start:.2f} s")
print("interior loss after training: ", pinn2.best_loss["interior"])
print("boundary loss after training: ", pinn2.best_loss["boundary"])
print("total    loss after training: ", pinn2.best_loss["total"])

Plot the two PINNs:


In [ ]:
import matplotlib.pyplot as plt
from scimba_jax.plots.plots_nd import plot_abstract_approx_spaces

plot_abstract_approx_spaces(
    (pinn.space, pinn2.space),  # the approximation spaces
    domain_x,  # the spatial domain
    domain_mu,  # the parameter's domain
    loss=(pinn.losses, pinn2.losses),  # for plot of the loss: the losses
    residual=(pinn.model, pinn2.model),  # for plot of the residual: the pde
    error=analytic_solution,  # for plot of the error with respect to a func: the func
    draw_contours=True,
    n_drawn_contours=20,
    titles=("with strong boundary conditions", "with weak boundary conditions"),
)
plt.show()

Feel free to experiment with the strong BC! For instance, change the post-processing function and check whether some choices are better suited than others (e.g. by using what we know about the exact solution of the PDE).


## 2/ Approximation of the solution of a 1D heat equation

Let us now demonstrate how to define and train a PINN for a time-dependent problem.

We consider the 1D heat equation

$$\partial_t u - \partial_{xx} u = 0 \text{ in } \Omega \times (0, T)$$

with boundary conditions:
$$u = 0 \text{ on } \partial \Omega \times (0, T)$$

and initial conditions:
$$u = u_0 \text{ on } \Omega \times {0}$$

where $u: \Omega \times (0, T) \to \mathbb{R}$ is the unknown function,
$\Omega \subset \mathbb{R}$ and $(0, T) \subset \mathbb{R}$.
Homogeneous Dirichlet boundary conditions are prescribed.

The exact solution reads
$$u(x,t) = \sin(\pi x) \exp(-\pi^2 t),$$
with the initial condition $u_0(x) = \sin(\pi x)$.

We will demonstrate how to approximate the solution of this problem with three approaches:

- strong boundary conditions and weak initial conditions enforcement,
- weak boundary conditions and strong initial conditions enforcement,
- strong boundary conditions and strong initial conditions enforcement.


We define the exact solution and the initial condition:


In [ ]:
def exact_sol(t: jnp.ndarray, x: jnp.ndarray) -> jnp.ndarray:
    """Exact solution of the 1D heat equation."""
    return jnp.exp(-(t * jnp.pi**2)) * jnp.sin(jnp.pi * x)


def f_init(x: jnp.ndarray) -> jnp.ndarray:
    """Initial condition of the 1D heat equation."""
    t = jnp.zeros_like(x)
    return exact_sol(t, x)

Define global variables and import additional modules:


In [ ]:
from scimba_jax.domains.meshless_domains.domains_1d import Segment1D
from scimba_jax.nonlinear_approximation.integration.monte_carlo_time import (
    UniformTimeSampler,
)
from scimba_jax.physical_models.temporal_pde.heat_equations import HeatND

N_COLLOC = 900
N_BC_COLLOC = 2000
N_IC_COLLOC = 2000
N_EPOCHS = 50
S_LAYERS = [16, 16]

We define the domains and the sampler.
Here the sampler is common to the three approaches, and therefore it will always generate samples for the boundary and initial conditions, which is unnecessary but fine.


In [ ]:
domain_t = (0.0, 0.5)
domain_x = Segment1D((0.0, 1.0), is_main_domain=True)

sampler = TensorizedSampler(
    [
        UniformTimeSampler(domain_t),
        DomainSampler(domain_x),
    ],
    model_type="t_x",
    bc=True,
    ic=True,
)

We now define the three post-processing functions corresponding to the three approaches demonstrated below:


In [ ]:
def post_processing_weak_ic_strong_bc(
    approx: jnp.ndarray, t: jnp.ndarray, x: jnp.ndarray
) -> jnp.ndarray:
    """Post-processing function to enforce weak IC, strong BC."""
    TODO("strong BC in x on [0, 1], weak IC: multiply by x * (1 - x)")


def post_processing_strong_ic_weak_bc(
    approx: jnp.ndarray, t: jnp.ndarray, x: jnp.ndarray
) -> jnp.ndarray:
    """Post-processing function to enforce strong IC, weak BC."""
    TODO("strong IC, weak BC: add the initial condition, and multiply the network output by t (so that it vanishes at t = 0)")


def post_processing_strong_ic_strong_bc(
    approx: jnp.ndarray, t: jnp.ndarray, x: jnp.ndarray
) -> jnp.ndarray:
    """Post-processing function to enforce strong IC, strong BC."""
    TODO("strong IC and strong BC: combine both tricks")

### Defining and training a PINN with strong boundary conditions and weak initial conditions


In [ ]:
model = HeatND(
    main_domain=domain_x,
    time_domain=domain_t,
    bc=TODO("strong or weak boundary conditions? (this approach: strong)"),
    ic=TODO("strong or weak initial conditions? (this approach: weak)"),
    f_ic_rhs=lambda *args: f_init(*args),
)

key = jax.random.PRNGKey(0)
nn1 = MLP(in_size=2, out_size=1, hidden_sizes=S_LAYERS, key=key)

space1 = ApproximationSpace(
    {"x": 1},
    [(nn1, "scalar", None)],
    model_type="t_x",
    post_processing=post_processing_weak_ic_strong_bc
)

pinn1 = Projector(model, space1, sampler)

start = timeit.default_timer()
key, pinn1 = pinn1.project(
    key, space1, N_EPOCHS, N_COLLOC, n_ic_colloc=N_IC_COLLOC
)
end = timeit.default_timer()

### Defining and training a PINN with weak boundary conditions and strong initial conditions


In [ ]:
model = HeatND(
    main_domain=domain_x,
    time_domain=domain_t,
    bc=TODO("weak boundary conditions here"),
    ic=TODO("strong initial conditions here"),
    f_ic_rhs=lambda *args: f_init(*args),
)

key = jax.random.PRNGKey(0)
nn2 = MLP(in_size=2, out_size=1, hidden_sizes=S_LAYERS, key=key)

space2 = ApproximationSpace(
    {"x": 1},
    [(nn2, "scalar", None)],
    model_type="t_x",
    post_processing=post_processing_strong_ic_weak_bc
)

pinn2 = Projector(model, space2, sampler)

start = timeit.default_timer()
key, pinn2 = pinn2.project(
    key, space2, N_EPOCHS, N_COLLOC, n_bc_colloc=N_BC_COLLOC
)
end = timeit.default_timer()

### Defining and training a PINN with strong boundary conditions and strong initial conditions


In [ ]:
model = HeatND(
    main_domain=domain_x,
    time_domain=domain_t,
    bc=TODO("strong boundary conditions"),
    ic=TODO("strong initial conditions"),
    f_ic_rhs=lambda *args: f_init(*args),
)

key = jax.random.PRNGKey(0)
nn3 = MLP(in_size=2, out_size=1, hidden_sizes=S_LAYERS, key=key)

space3 = ApproximationSpace(
    {"x": 1},
    [(nn3, "scalar", None)],
    model_type="t_x",
    post_processing=post_processing_strong_ic_strong_bc
)

pinn3 = Projector(model, space3, sampler)

start = timeit.default_timer()
key, pinn3 = pinn3.project(
    key, space3, N_EPOCHS, N_COLLOC
)
end = timeit.default_timer()

### Compare the results


In [ ]:
plot_abstract_approx_spaces(
    (pinn1.space, pinn2.space, pinn3.space),
    domain_x,
    time_domains=domain_t,
    loss=(pinn1.losses, pinn2.losses, pinn3.losses),
    residual=(pinn1.model, pinn2.model, pinn3.model),
    solution=exact_sol,
    error=exact_sol,
    title="learning sol of 1D heat equation with PINNs",
    titles=("weak IC strong BC", "strong IC weak BC", "strong IC strong BC"),
)

plt.show()